# ⏱️ trend_following_1m — the trend follower on 1-minute bars of ten liquid stocks

**The question.** Daily bars are coarse for a trend follower: a breakout is only seen at the close and a stop only checked once a day. This notebook runs **the same strategy code** as [`trend_following`](../trend_following/README.md) — Donchian + ADX start, ratcheting ATR chandelier, opposite-start exit, equal slots — on **1-minute bars**, where every window is in minutes, with two intraday rules: no starts in the first/last `skip_open`/`skip_close` minutes of a session, and `eod_exit` (flat before each session's last bar, or hold overnight — searched).

**The stocks.** `Morning10`: HOOD MU COIN SMCI INTC ORCL GS NOW MRVL SPOT — the ten most liquid names of `Morning200`, ranked by median dollar volume over 2025-01..06, **before** the 1-minute window opens. One slot per stock (10 × 10%).

**The windows.** 1-minute history 2025-06-18 → 2026-06-26. The study trains on 2025-07-01 → 2026-04-29 (scored warm), the **holdout** is 2026-04-30 → 2026-06-26, read once (§12b). The objective is `alpha_cagr` against volatility: buy-and-hold of these ten names made +54% over the ten train months and +27% in the two holdout months, all of it overnight, so an excess return over it would condemn any intraday book regardless of selection. Full write-up, including the same study on 5- and 15-minute bars: [README.md](./README.md).

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

strat = strategies.load("trend_following_1m")
compose = strat.signals.__globals__["compose"]   # trend_following's intermediates
d = strat.data
SCORE_FROM = strat.optuna["score_from"]
INIT_CASH = settings.backtest.init_cash
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges,
                   resample=d.get("resample"))
print(f"{d['universe']}: {', '.join(syms)}")
print(f"{panel.close.shape[0]:,} one-minute bars {panel.close.index[0]} → {panel.close.index[-1]}"
      f" (scored from {SCORE_FROM})")
print("objectives:", list(opt.objectives_for(strat)), "| order notional:", strat.order_notional,
      "| study:", strat.optuna["study"])

## 2 · Strategy overview — hyperparameters and search ranges

| param | role | intuition on 1-minute bars |
|---|---|---|
| `entry_window` | Donchian lookback of the start (minutes) | 390 = one session |
| `adx_window`, `adx_min` | ADX confirmation (±DI agreeing) | only directional breakouts start a trend |
| `atr_window`, `atr_mult` | the chandelier: distance in minute-ATRs, current ATR, ratcheted | a 1m ATR is ~0.1–0.2% of price |
| `eod_exit` | flat before the close, or hold overnight | **the** structural choice — searched |
| `skip_open`, `skip_close` | entry blackouts (minutes) | no breakouts of the overnight gap, none into the close |
| `direction` | long / short / both | grid ablation |
| `max_positions` | slots — one per stock | the relative-strength admission (`rs_*`) is inert with a slot each |

In [ ]:
def _fmt_space(spec):
    # One readable line for an optuna.space entry.
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "shipped": pd.Series(strat.params),
    "ablation grid (§6)": pd.Series({k: ", ".join(map(str, v))
                                     for k, v in strat.sweep.items()}),
    "optuna search range (§7)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · One backtest of the shipped config — and the baseline it has to beat

Scored **warm** from `SCORE_FROM`. The baseline is the equal-weight buy-and-hold of the same ten stocks (`strategies/baseline`). Read `alpha_cagr` beside `beta` — this book is flat overnight unless `eod_exit` is off.

In [ ]:
pf = bt.run_strategy(strat, panel, settings=settings, score_from=SCORE_FROM)
stats0 = bt.evaluate(strat, panel, {}, settings=settings, score_from=SCORE_FROM)
display(pd.Series(stats0).astype(float).round(4).to_frame("shipped (train, warm)"))
pf_base = bt.run_strategy(strategies.load("baseline"), panel, settings=settings,
                          score_from=SCORE_FROM)
fig = go.Figure()
for p, name, color, dash in ((pf_base, "baseline (buy & hold)", "#898781", "dash"),
                             (pf, strat.name, plots.SERIES[0], None)):
    curve = bt.equity_curve(p, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0
    fig.add_trace(plots._scatter(
        curve, name=name, line={"color": color, "dash": dash, "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + name + "</extra>"))
plots._base(fig, title="train — cumulative return vs equal-weight buy-and-hold", height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 4 · Event test — does an intraday trend start carry information worth its costs?

Every start at a grid of channel lengths, with and without the ADX confirmation, scored by its forward return from the next minute's open over 5 … 120 minutes **inside the same session**, signed for the trade, in basis points. `t` is day-clustered (starts overlap in time). A round trip costs **~8 bp** here (2 bp commission at $5,000 orders + 1 bp half-spread + 1 bp slippage, per leg) — a start has to move the price by more than that, on average, before any exit logic can make it pay.

Then the fact that decides the `eod_exit` question: where did these stocks make their money — overnight or during the session?

In [ ]:
idx = panel.close.index
day = pd.Series(idx.tz_convert(panel.tz).date, index=idx)
lo = pd.Timestamp(SCORE_FROM, tz="UTC")
HORIZONS = (5, 15, 30, 60, 120)
fwd = {}
for h in HORIZONS:
    f = panel.close.shift(-h) / panel.open_.shift(-1) - 1.0
    same = (day.shift(-h) == day).to_numpy()
    fwd[h] = f.where(np.repeat(same[:, None], f.shape[1], axis=1)).loc[lo:]

rows = []
for ew in (30, 120, 390):
    for amin in (0.0, 30.0, 57.0):
        parts = compose(panel, **{**strat.params, "entry_window": ew, "adx_min": amin})
        for side, key, sign in (("long", "long_start", 1), ("short", "short_start", -1)):
            mask = parts[key].loc[lo:]
            row = {"entry_window": ew, "adx_min": amin, "side": side,
                   "starts/day": mask.to_numpy().sum() / day.loc[lo:].nunique()}
            for h in HORIZONS:
                ev = (sign * fwd[h]).where(mask).stack()
                mu = ev.mean()
                dev = (ev - mu).groupby(ev.index.get_level_values(0).date).sum()
                row[f"{h}m bp"] = mu * 1e4
                row[f"{h}m t"] = mu / (np.sqrt((dev ** 2).sum()) / len(ev))
            rows.append(row)
display(pd.DataFrame(rows).round(1))

ret = panel.close.pct_change(fill_method=None).loc[lo:]
first = (day.loc[lo:] != day.loc[lo:].shift(1)).to_numpy()
overnight = np.log1p(ret[first]).mean().mean() * 1e4
intraday = np.log1p(ret[~first]).groupby(day.loc[lo:][~first]).sum().mean().mean() * 1e4
print(f"average day, per stock: overnight gap {overnight:+.1f} bp, the session itself "
      f"{intraday:+.1f} bp — a book that is flat at the close gives up the whole overnight leg")

## 5 · Inspect a symbol — one week of minutes

The chandelier actually in force (NaN when flat), the Donchian channel, ▲/▼ entries and × exits, zoomed to the week around the stock's busiest trading day so the minutes are readable. Used again in §11 on the optimized pick.

In [ ]:
def inspect_symbol(sym, pf, params, days=5):
    # One symbol, one week: annotated chart + that week's trades.
    merged = {**strat.params, **params}
    trades = bt.trade_records(pf)
    mine = trades[trades["symbol"] == sym]
    busiest = mine["entry_ts"].dt.tz_convert(panel.tz).dt.date.value_counts().index[0]
    local = pf.wrapper.index.tz_convert(panel.tz).date
    dates = sorted(set(local))
    at = dates.index(busiest)
    keep = set(dates[max(0, at - days // 2): at + days // 2 + 1])
    window = pf.wrapper.index[[x in keep for x in local]]
    parts = compose(panel, **merged)
    overlays = strat.indicators(panel.close[sym], panel.volume[sym], **merged)
    overlays = {**overlays, "price": {**{k: v.loc[window] for k, v in overlays["price"].items()},
                                      "chandelier stop": parts["stop"][sym].loc[window]}}
    shown = mine[mine["entry_ts"].isin(window) | mine["exit_ts"].isin(window)]
    plots.plot_symbol(panel.close[sym].loc[window], panel.volume[sym].loc[window], shown,
                      symbol=sym, overlays=overlays).show()
    return shown.reset_index(drop=True).round(3)

trades0 = bt.trade_records(pf)
inspect_symbol(trades0["symbol"].value_counts().index[0], pf, {})

## 6 · Ablation grid — which side, flat at the close or not, and how strict a start?

`direction` × `eod_exit` × `adx_min` (a moderate 25 against the shipped 57) — 12 combos, everything else shipped, each scored warm through `bt.evaluate` exactly like a study trial. The ADX floor is the knob that decides everything on minute bars: it sets how rarely the book trades.

> **Read `cagr` beside `alpha_cagr` on the one-sided rows.** Alpha is `cagr − beta × benchmark cagr`, so a book that is net SHORT (negative beta) is credited for the whole market rise it stood against: a short-only book that holds overnight can lose money and still post a large positive alpha. The shipped book has beta ≈ 0, where alpha ≈ its own return; a negative-beta row's alpha is a statement about the benchmark, not about selection.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
rows = []
for combo in grid:
    st = bt.evaluate(strat, panel, combo, settings=settings, score_from=SCORE_FROM)
    rows.append({**combo, **st})
sweep_stats = pd.DataFrame(rows)
cols = ["direction", "eod_exit", "adx_min", "alpha_cagr", "beta", "cagr", "volatility",
        "max_drawdown", "trades_per_year", "win_rate"]
display(sweep_stats[cols].sort_values("alpha_cagr", ascending=False).round(3))
sweep_stats["eod_exit"] = sweep_stats["eod_exit"].map({True: "flat at close", False: "overnight"})
plots.plot_heatmap_grid(sweep_stats, pairs=[("direction", "adx_min"), ("eod_exit", "adx_min")],
                        metrics=("alpha_cagr", "volatility")).show()

## 7 · Optimize (resumable, runs outside the notebook)

Tops the study up to `optuna.n_trials` in a `tsim optimize` subprocess — re-running this cell never adds trials past it. One trial is a year of minutes (~5 s).

In [ ]:
path = opt.study_path(strat, strat.optuna["study"])
done = len(opt.load_study(strat).trials) if path.exists() else 0
missing = max(0, int(strat.optuna["n_trials"]) - done)
print(f"study {strat.optuna['study']}: {done} trials on disk, running {missing} more")
if missing:
    jobs.run_optimize("trend_following_1m", n_trials=missing)

## 8 · Read the study — the pick

`opt.pick_from_front`: only front rows with `alpha_cagr` > 0 (the study's hurdle) are eligible; if none clears it the pick is flagged `beats_benchmark = False` and everything downstream is a diagnosis, not a strategy.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)
space = strat.optuna["space"]
_INTS = {k for k, s in space.items() if s["type"] == "int"}

def coerce(row):
    # Cast a front row back to real parameter types.
    return {k: (int(round(float(row[k]))) if k in _INTS
                else bool(row[k]) if space[k]["type"] == "categorical" else float(row[k]))
            for k in space}

best_row = opt.pick_from_front(front)
picked = coerce(best_row)
print(f"trials: {len(study.trials)} total, {len(everything)} scored, "
      f"{int((everything['alpha_cagr'] > 0).sum())} with positive alpha; "
      f"front: {len(front)} rows, {int(best_row['front_distinct'])} distinct; "
      f"beats its hurdle: {bool(best_row['beats_benchmark'])}")
print("picked params:", picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4) for m in
      ("alpha_cagr", "volatility", "beta", "cagr", "max_drawdown", "trades_per_year")})
print("scored trials by eod_exit (alpha_cagr):")
display(everything.groupby("eod_exit")["alpha_cagr"].describe().round(3))
plots.plot_pareto(front, everything, picked=int(best_row["trial"])).show()

## 9 · Boundary check — and what lies past the ADX bound

The pick's ADX floor (57) sits within 5% of the searched bound (60), so the report says *widen high*. Before bumping the study, the cheap question: re-score the pick with the floor **past** the bound, on the train window only. The trade rate is the constraint — `optuna.min_trades_per_year` (250) prunes anything that trades less.

In [ ]:
report = opt.boundary_report(picked, space, front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
display(report)
rows = {}
for floor in (50.0, 54.0, picked["adx_min"], 60.0, 65.0, 70.0):
    st = bt.evaluate(strat, panel, {**picked, "adx_min": floor}, settings=settings,
                     score_from=SCORE_FROM)
    rows[floor] = {"alpha_cagr": st["alpha_cagr"], "trades_per_year": st["trades_per_year"],
                   "volatility": st["volatility"]}
display(pd.DataFrame(rows).T.rename_axis("adx_min").round(4))

## 10 · Stability — plateau or spike?

Each picked value perturbed by ±5% / ±10% of its searched span (and the other `eod_exit`), one at a time, scored warm on the train window.

In [ ]:
star = opt.neighbors(picked, space, valid=lambda p: strat.valid({**strat.params, **p}))
rows = []
for rec in star.to_dict("records"):
    p = {k: (int(rec[k]) if k in _INTS else rec[k] if space[k]["type"] == "categorical"
             else float(rec[k])) for k in space}
    st = bt.evaluate(strat, panel, p, settings=settings, score_from=SCORE_FROM)
    rows.append({**p, "perturbed": rec["perturbed"], "offset": rec["offset"], **st})
star = pd.DataFrame(rows)
moved = star[star["perturbed"] != ""]
print(f"{len(moved)} neighbours: {int((moved['alpha_cagr'] > 0).sum())} with positive alpha, "
      f"median {moved['alpha_cagr'].median():+.2%}/yr "
      f"(centre {star.loc[star['perturbed'] == '', 'alpha_cagr'].iloc[0]:+.2%})")
display(opt.stability_report(star, metric="alpha_cagr"))
plots.plot_sensitivity(star[star["offset"].notna() | (star["perturbed"] == "")],
                       metric="alpha_cagr").show()

## 11 · Backtest the picked parameters — and verify them symbol by symbol

In [ ]:
pf_best = bt.run_strategy(strat, panel, params=picked, settings=settings, score_from=SCORE_FROM)
stats_best = bt.evaluate(strat, panel, picked, settings=settings, score_from=SCORE_FROM)
display(pd.Series(stats_best).astype(float).round(4).to_frame("picked (train, warm)"))
plots.plot_equity(bt.equity_curve(pf_best, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity with picked parameters").show()
display(bt.symbol_stats(pf_best).round(4))

In [ ]:
SYMBOL = None   # ← any symbol above; None = the most traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, picked)

## 12 · Validate

### 12a · Month by month (train)

Ten months of one-minute trading, each its own warm-scored book against its own buy-and-hold. Alpha and beta are **not annualized away**: `period_return` and `bh_return` are the plain returns of the month.

In [ ]:
def upto(p, end):
    # The panel up to `end` (a timestamp, inclusive) — used only to END a window.
    ts = pd.Timestamp(end)
    ts = ts.tz_localize("UTC") if ts.tzinfo is None else ts

    def cut(frame):
        return frame.loc[frame.index <= ts]

    return replace(p, close=cut(p.close), open_=cut(p.open_), high=cut(p.high),
                   low=cut(p.low), volume=cut(p.volume))

full = load_panel(engine, syms, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges,
                  resample=d.get("resample"))
# The ten stocks' stored 1m histories do not end together (MU runs on to September): stop at
# the last bar EVERY one of them has, or the tail is a one-stock book against a one-stock benchmark.
complete = full.close.notna().all(axis=1)
full = upto(full, complete[complete].index[-1])
HOLDOUT_START = "2026-04-30"      # the first session the train panel (end 2026-04-30) excludes
LAST = str(full.close.index[-1].date())
print(f"full history: {full.close.shape[0]:,} bars → {LAST} (the last bar all ten stocks share)")

def period(p, params, start, end, strategy=strat):
    # Warm-scored stats plus the PLAIN (not annualized) returns of the window.
    st = bt.evaluate(strategy, upto(p, end), params, settings=settings, score_from=start)
    growth = (1 + st["total_return"]), (1 + st["cagr"] - st["excess_cagr"]) ** st["years"]
    return {"period_return": growth[0] - 1, "bh_return": growth[1] - 1,
            "beta": st["beta"], "alpha_cagr": st["alpha_cagr"], "volatility": st["volatility"],
            "max_drawdown": st["max_drawdown"], "trades": st["n_trades"]}

months = pd.period_range("2025-07", "2026-04", freq="M")
table = {}
for m in months:
    start, end = str(m.start_time.date()), f"{m.end_time.date()} 23:59"
    if m == months[-1]:
        end = "2026-04-29 23:59"
    table[str(m)] = period(full, picked, start, end)
display(pd.DataFrame(table).T.astype(float).round(3))

### 12b · The untouched holdout (open once)

2026-04-30 → 2026-06-26 was never seen by the study. Two months of minutes annualize violently — read the plain period returns first. **Pass** = holdout `alpha_cagr` > 0 **and** the book's daily P&L distinguishable from zero (t > 2 over the holdout's days). The information ratio is printed but not a gate: against a +27% two-month rally it is negative for any book that is flat overnight, whatever it selects.

In [ ]:
st_hold = bt.evaluate(strat, full, picked, settings=settings, score_from=HOLDOUT_START)
plain = period(full, picked, HOLDOUT_START, f"{LAST} 23:59")
print(f"HOLDOUT {HOLDOUT_START} → {LAST} ({st_hold['years'] * 252:.0f} sessions, "
      f"{int(st_hold['n_trades'])} trades)")
print(f"  strategy   {plain['period_return']:+.2%} over the period   beta {st_hold['beta']:+.2f}   "
      f"max drawdown {st_hold['max_drawdown']:.1%}")
print(f"  buy & hold {plain['bh_return']:+.2%} over the period")
pf_hold = bt.run_strategy(strat, full, params=picked, settings=settings, score_from=HOLDOUT_START)
value = bt.equity_curve(pf_hold)["value"]
daily_pnl = value.groupby(value.index.tz_convert(full.tz).date).last().pct_change().dropna()
t_day = daily_pnl.mean() / daily_pnl.std() * np.sqrt(len(daily_pnl))
print(f"  ALPHA      {st_hold['alpha_cagr']:+.2%}/yr (annualized)   daily P&L "
      f"{daily_pnl.mean() * 1e4:+.1f} bp, t {t_day:+.2f} over {len(daily_pnl)} days   "
      f"(information ratio {st_hold['info_ratio']:+.2f})")
verdict = ("PASS" if st_hold["alpha_cagr"] > 0 and t_day > 2 else
           "POSITIVE BUT NOT SIGNIFICANT" if st_hold["alpha_cagr"] > 0 else "FAIL")
print("  -->", verdict, "on the holdout")

pf_bh = bt.run_strategy(strategies.load("baseline"), full, settings=settings,
                        score_from=HOLDOUT_START)
fig = go.Figure()
for p, name, color, dash in ((pf_bh, "baseline (buy & hold)", "#898781", "dash"),
                             (pf_hold, strat.name, plots.SERIES[0], None)):
    curve = bt.equity_curve(p, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0
    fig.add_trace(plots._scatter(
        curve, name=name, line={"color": color, "dash": dash, "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + name + "</extra>"))
plots._base(fig, title="HOLDOUT — cumulative return vs equal-weight buy-and-hold", height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

### 12c · Costs and fills

Friction at the repo's two fixed bars (6/5 and 10/3 bps spread/slippage per leg), then **pessimistic fills** (every buy at the minute's high, every sell at its low), on train and holdout.

In [ ]:
def with_costs(**overrides):
    # A copy of settings with its (pydantic) cost block overridden.
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

rows = {}
for label, s, st_ in (("base costs", settings, strat),
                      ("6/5 bps", with_costs(spread_bps=6.0, slippage_bps=5.0), strat),
                      ("10/3 bps", with_costs(spread_bps=10.0, slippage_bps=3.0), strat),
                      ("pessimistic fills", settings, replace(strat, fill="pessimistic"))):
    for window, p, start in (("train", upto(full, "2026-04-29 23:59"), SCORE_FROM),
                             ("holdout", full, HOLDOUT_START)):
        st = bt.evaluate(st_, p, picked, settings=s, score_from=start)
        rows[(label, window)] = {"alpha_cagr": st["alpha_cagr"], "cagr": st["cagr"]}
pd.DataFrame(rows).T.unstack().round(4)

## 13 · One minute or one day? — the same ten stocks, the same windows

The daily strategy (`trend_following`, its shipped parameters, never fit on these names) on the same ten stocks' daily bars, with one slot per stock and the same $5,000-order cost model, over the same train and holdout windows — beside the one-minute pick.

In [ ]:
daily = strategies.load("trend_following")
daily10 = replace(daily, order_notional=strat.order_notional)
dpanel = load_panel(engine, syms, "1d", "2024-01-01", None, session="regular",
                    exchanges=settings.exchanges)
rows = {}
for label, (p1m, pd1, start, end) in {
        "train": (full, dpanel, SCORE_FROM, "2026-04-29 23:59"),
        "holdout": (full, dpanel, HOLDOUT_START, f"{LAST} 23:59")}.items():
    rows[("1-minute (picked)", label)] = period(p1m, picked, start, end)
    rows[("daily (shipped)", label)] = period(pd1, {**daily.params, "max_positions": 10},
                                              start, end, strategy=daily10)
pd.DataFrame(rows).T.astype(float).round(3)

## Read the result honestly

* **§4 is the verdict before the study.** An intraday trend start moves the price by a few basis points either way over the next two hours — statistically nothing (every day-clustered |t| ≤ 1.2) — against ~8 bp of round-trip cost. What the study can still find is a *rare* start that happens to pay, and it did: the pick trades only the most violent bursts (ADX ≥ 57 on minute bars) and is in the market about a tenth of the time.
* **Positive, but not distinguishable from luck.** The pick earns a few basis points a day on train and on the holdout, with near-zero beta — and a day-level t below 1 in both. It is the best of 300 trials, which makes the in-sample number weaker still. At 6/5 bps of friction the train alpha is almost gone; with pessimistic fills it is negative.
* **Where the money was.** These stocks made their gains overnight and gave some back during the session. A book that is flat at the close forfeits the one leg that paid; the study's highest-alpha rows hold overnight — which is exposure, not intraday trend skill.
* **More bars is not more information.** The same study on 5- and 15-minute bars (README) tells the same story: small, insignificant alphas, carried by one or two names. The daily version on these ten stocks is no better (alpha ≈ 0). Nothing here comes close to simply owning the ten stocks, which is a statement about this window (+54%, then +27%) as much as about the strategy.
* **Short windows.** Ten months of train and two of holdout, on ten correlated tech/fintech names. Breadth (Morning200) is the next thing to try, not a finer bar — see IDEAS.md.